# RQ3 Supplementary Information: Clean Robustness and Diagnostics

This notebook generates a compact SI package for RQ3. It is intentionally narrower than the model-selection notebooks: outputs are limited to diagnostics and robustness evidence that directly support the selected main specification.

SI contents:

0. Core pretest diagnostics: sample, correlations, and VIF.
1. Global model stability across four factor-selection specifications.
2. Spatial residual diagnostics and spatial error sensitivity for the selected M4 model.
3. Fractional logit robustness for `sufficiency_gap` and `sufficiency_deficit`, including global and regional models.
4. Complete pre-specified interaction results for the selected M4 outcome, including FDR q-values and simple slopes for focused female-share interactions.

In [ ]:

from pathlib import Path
import json
import sys
import warnings

PROJECT = Path('/Users/nico/Research/Energy sufficiency')
LOCAL_PKG = PROJECT / 'Process/python_packages'
if LOCAL_PKG.exists() and str(LOCAL_PKG) not in sys.path:
    sys.path.insert(0, str(LOCAL_PKG))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', font='DejaVu Sans')

DATA_PATH = PROJECT / 'Data/global_SE_EC_sufficiency_controls_2015.csv'
SI_DIR = PROJECT / 'Results/RQ3_social_determinants/SI'
SI_TABLE_DIR = SI_DIR / 'Tables'
SI_FIG_DIR = SI_DIR / 'Figures'
PROCESS_DIR = PROJECT / 'Process/RQ3_SI_clean'
for d in [SI_DIR, SI_TABLE_DIR, SI_FIG_DIR, PROCESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Data:', DATA_PATH)
print('SI results:', SI_DIR)


In [ ]:

def star(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.1: return '†'
    return ''

def pct_change(beta):
    return (np.exp(beta) - 1) * 100

def clean_model_data(df):
    d = df.copy()
    d['income_clean'] = d['income'].where(d['income'] >= 0, np.nan)
    d['wealth_clean'] = d['wealth'].where(d['wealth'] >= 0, np.nan)
    d['SES_raw'] = d['income_clean'].combine_first(d['wealth_clean'])
    d['female_share'] = d['gender']
    d['log_sufficiency_ratio'] = np.log(d['sufficiency_ratio'].where(d['sufficiency_ratio'] > 0))
    d['GDP_pc'] = d['GDP'] / d['population'].replace(0, np.nan)
    d['log_GDP_pc'] = np.log(d['GDP_pc'].where(d['GDP_pc'] > 0))
    d['log_population'] = np.log(d['population'].where(d['population'] > 0))
    return d

raw = pd.read_csv(DATA_PATH)
df = clean_model_data(raw)

controls = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
core_predictors = ['SES_raw', 'rural', 'education', 'female_share']
M1_predictors = core_predictors + ['age', 'Hhsize'] + controls
M2_predictors = core_predictors + ['Hhsize'] + controls
M3_predictors = core_predictors + ['age'] + controls
M4_predictors = core_predictors + controls
model_families = {
    'M1_full_age_hhsize': M1_predictors,
    'M2_drop_age': M2_predictors,
    'M3_drop_hhsize': M3_predictors,
    'M4_drop_age_hhsize': M4_predictors,
}

all_needed = ['GDLcode', 'iso_code', 'country', 'analytical_region', 'population', 'log_sufficiency_ratio', 'sufficiency_gap', 'sufficiency_deficit'] + M1_predictors
sample = df.dropna(subset=all_needed).copy()
sample = sample[(sample['population'] > 0) & np.isfinite(sample['log_sufficiency_ratio'])].copy()
sample = sample[(sample['sufficiency_gap'].between(0, 1)) & (sample['sufficiency_deficit'].between(0, 1))].copy()
std = sample.copy()
for var in sorted(set(M1_predictors)):
    std['z_' + var] = (std[var] - std[var].mean()) / std[var].std(ddof=0)

sample_summary = pd.DataFrame([
    {'metric': 'Raw GDL observations', 'value': len(raw)},
    {'metric': 'Complete RQ3 SI/common model sample', 'value': len(std)},
    {'metric': 'Countries in sample', 'value': std['iso_code'].nunique()},
    {'metric': 'Analytical regions in sample', 'value': std['analytical_region'].nunique()},
])
sample_summary.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_00_sample_summary.csv', index=False)
print(sample_summary)


In [ ]:

# 0. Core pretest diagnostics: correlations and VIF for main M4 variables.
main_vars = ['log_sufficiency_ratio'] + M4_predictors
label_map = {
    'log_sufficiency_ratio': 'Log sufficiency ratio', 'SES_raw': 'SES', 'rural': 'Rurality',
    'education': 'Education', 'female_share': 'Female share', 'log_GDP_pc': 'GDP per capita',
    'log_population': 'Population', 'total_HDD': 'Heating degree days', 'total_CDD': 'Cooling degree days'
}

corr = std[main_vars].corr(method='pearson')
pvals = pd.DataFrame(np.ones_like(corr), index=main_vars, columns=main_vars)
for i, a in enumerate(main_vars):
    for j, b in enumerate(main_vars):
        if i == j:
            pvals.loc[a, b] = 0
        else:
            r, p = stats.pearsonr(std[a], std[b])
            pvals.loc[a, b] = p
corr.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_00_main_variable_correlations_r.csv')
pvals.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_00_main_variable_correlations_p.csv')

annot = corr.copy().astype(str)
for a in main_vars:
    for b in main_vars:
        stars = star(pvals.loc[a, b]) if a != b else ''
        annot.loc[a, b] = f'{corr.loc[a, b]:.2f}{stars}' if a != b else '1.00'
fig, ax = plt.subplots(figsize=(8.2, 6.8), constrained_layout=True)
sns.heatmap(corr.rename(index=label_map, columns=label_map), annot=annot.rename(index=label_map, columns=label_map), fmt='', cmap='vlag', center=0, vmin=-1, vmax=1, square=True, linewidths=.3, cbar_kws={'label': 'Pearson r'}, ax=ax)
ax.set_title('RQ3 main variables: Pearson correlations')
fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_00_main_variable_correlation_heatmap.png', bbox_inches='tight')
plt.show()

X = sm.add_constant(std[['z_' + v for v in M4_predictors]])
vif = []
for i, col in enumerate(X.columns):
    if col == 'const':
        continue
    vif.append({'variable': col.replace('z_', ''), 'VIF': variance_inflation_factor(X.values, i)})
vif = pd.DataFrame(vif).sort_values('VIF', ascending=False)
vif['label'] = vif['variable'].map(label_map)
vif.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_00_main_M4_VIF.csv', index=False)
fig, ax = plt.subplots(figsize=(6.5, 3.8), constrained_layout=True)
sns.barplot(data=vif, x='VIF', y='label', color='#4c78a8', ax=ax)
ax.axvline(5, color='#d95f02', linestyle='--', linewidth=1)
ax.set_xlabel('Variance inflation factor')
ax.set_ylabel('')
ax.set_title('VIF diagnostics for selected M4 specification')
fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_00_main_M4_VIF.png', bbox_inches='tight')
plt.show()


In [ ]:

# Shared model functions.
def fit_fe_ols(data, predictors, outcome='log_sufficiency_ratio'):
    rhs = ' + '.join(['z_' + v for v in predictors]) + ' + C(iso_code)'
    return smf.ols(outcome + ' ~ ' + rhs, data=data).fit(cov_type='cluster', cov_kwds={'groups': data['iso_code']})

def extract_ols_rows(fit, data, model, predictors, outcome='log_sufficiency_ratio', region=None):
    rows=[]
    ci = fit.conf_int()
    for var in predictors:
        param = 'z_' + var
        b=float(fit.params[param]); se=float(fit.bse[param]); p=float(fit.pvalues[param])
        lo, hi = ci.loc[param].tolist()
        rows.append({
            'outcome': outcome, 'model': model, 'region': region, 'variable': var,
            'coef_log_points_per_sd': b, 'cluster_se': se, 'ci_low': float(lo), 'ci_high': float(hi),
            'p_value': p, 'stars': star(p), 'pct_change_per_sd': pct_change(b),
            'pct_ci_low': pct_change(lo), 'pct_ci_high': pct_change(hi),
            'n_obs': int(fit.nobs), 'n_countries': int(data['iso_code'].nunique()),
            'r_squared': float(fit.rsquared), 'adj_r_squared': float(fit.rsquared_adj),
            'aic': float(fit.aic), 'bic': float(fit.bic)
        })
    return rows

def fit_fractional_logit(data, outcome, predictors):
    rhs = ' + '.join(['z_' + v for v in predictors]) + ' + C(iso_code)'
    return smf.glm(outcome + ' ~ ' + rhs, data=data, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': data['iso_code']}, maxiter=300)

def extract_frac_rows(fit, data, model, outcome, predictors, region=None):
    rows=[]
    ci = fit.conf_int()
    mu = fit.predict(data)
    factor = float(np.mean(mu * (1 - mu)))
    for var in predictors:
        param = 'z_' + var
        b=float(fit.params[param]); se=float(fit.bse[param]); p=float(fit.pvalues[param])
        lo, hi = ci.loc[param].tolist()
        rows.append({
            'outcome': outcome, 'model': model, 'region': region, 'variable': var,
            'logit_coef_per_sd': b, 'cluster_se': se, 'ci_low': float(lo), 'ci_high': float(hi),
            'p_value': p, 'stars': star(p), 'AME_per_sd': b * factor,
            'AME_ci_low_approx': float(lo) * factor, 'AME_ci_high_approx': float(hi) * factor,
            'n_obs': int(fit.nobs), 'n_countries': int(data['iso_code'].nunique()),
            'aic': float(fit.aic), 'bic_llf': float(getattr(fit, 'bic_llf', np.nan))
        })
    return rows


In [ ]:

# 1. Global model stability across four factor-selection specifications.
stability_rows=[]
stability_summary=[]
for model, predictors in model_families.items():
    fit = fit_fe_ols(std, predictors)
    stability_rows.extend(extract_ols_rows(fit, std, model, predictors))
    stability_summary.append({
        'model': model, 'n_obs': int(fit.nobs), 'n_countries': int(std['iso_code'].nunique()),
        'r_squared': float(fit.rsquared), 'adj_r_squared': float(fit.rsquared_adj),
        'aic': float(fit.aic), 'bic': float(fit.bic)
    })
stability = pd.DataFrame(stability_rows)
stability_summary = pd.DataFrame(stability_summary)
stability.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_01_global_model_stability_coefficients.csv', index=False)
stability_summary.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_01_global_model_stability_summary.csv', index=False)

core = stability[stability['variable'].isin(core_predictors)].copy()
core['variable_label'] = core['variable'].map(label_map)
fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
sns.pointplot(data=core, x='pct_change_per_sd', y='variable_label', hue='model', dodge=.45, join=False, errorbar=None, ax=ax)
for _, row in core.iterrows():
    ax.plot([row['pct_ci_low'], row['pct_ci_high']], [list(core['variable_label'].drop_duplicates()).index(row['variable_label'])]*2, color='0.65', lw=.7, zorder=0)
ax.axvline(0, color='#333333', lw=.8)
ax.set_xlabel('Percent change in sufficiency ratio per 1 SD increase')
ax.set_ylabel('')
ax.set_title('Global model stability across factor-selection specifications')
fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_01_global_model_stability_core_coefficients.png', bbox_inches='tight')
plt.show()
stability[stability['variable'].isin(core_predictors)].sort_values(['variable','model'])


In [ ]:

# 2. Spatial residual diagnostics and spatial error sensitivity for selected M4.
spatial_status = {'status': 'not_run'}
try:
    import geopandas as gpd
    from libpysal.weights import Queen, KNN
    from esda.moran import Moran
    import spreg

    shp_path = Path('/Users/nico/Documents/PhD research/energy poverty/Data/GDL Shapefiles V4/GDL Shapefiles V4.shp')
    gdf = gpd.read_file(shp_path, columns=['GDLcode', 'geometry'])

    ols_fit = fit_fe_ols(std, M4_predictors)
    residuals = std[['GDLcode', 'iso_code', 'log_sufficiency_ratio'] + ['z_' + v for v in M4_predictors]].copy()
    residuals['fe_residual'] = ols_fit.resid
    sgdf = gdf.merge(residuals, on='GDLcode', how='inner').dropna(subset=['geometry', 'fe_residual']).reset_index(drop=True)

    wq = Queen.from_dataframe(sgdf, use_index=False)
    wq.transform = 'r'
    moran_q = Moran(sgdf['fe_residual'].values, wq, permutations=999, two_tailed=True)
    wk = KNN.from_dataframe(sgdf, k=6)
    wk.transform = 'r'
    moran_k = Moran(sgdf['fe_residual'].values, wk, permutations=999, two_tailed=True)
    moran_table = pd.DataFrame([
        {'model': 'M4 country-FE OLS residuals', 'weights': 'Queen contiguity', 'n_obs': len(sgdf), 'islands_or_components': len(wq.islands), 'moran_I': moran_q.I, 'expected_I': moran_q.EI, 'p_norm': moran_q.p_norm, 'p_sim': moran_q.p_sim},
        {'model': 'M4 country-FE OLS residuals', 'weights': 'KNN k=6', 'n_obs': len(sgdf), 'islands_or_components': 0, 'moran_I': moran_k.I, 'expected_I': moran_k.EI, 'p_norm': moran_k.p_norm, 'p_sim': moran_k.p_sim},
    ])
    moran_table.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_02_spatial_residual_morans_I.csv', index=False)

    y = sgdf[['log_sufficiency_ratio']].values
    X_base = sgdf[['z_' + v for v in M4_predictors]].reset_index(drop=True)
    country_dummies = pd.get_dummies(sgdf['iso_code'], prefix='cty', drop_first=True).reset_index(drop=True)
    X = pd.concat([X_base, country_dummies], axis=1).astype(float)
    sem = spreg.ML_Error(y, X.values, w=wk, name_y='log_sufficiency_ratio', name_x=list(X.columns), method='LU', epsilon=1e-7)

    sem_rows=[]
    name_to_idx = {name: i for i, name in enumerate(sem.name_x)}
    for var in M4_predictors:
        name = 'z_' + var
        idx = name_to_idx[name]
        b = float(sem.betas[idx, 0])
        se = float(sem.std_err[idx])
        z_stat, p = sem.z_stat[idx]
        sem_rows.append({
            'model': 'Spatial error sensitivity, ML_Error KNN k=6', 'variable': var,
            'coef_log_points_per_sd': b, 'std_error': se, 'z_stat': float(z_stat), 'p_value': float(p), 'stars': star(float(p)),
            'pct_change_per_sd': pct_change(b), 'n_obs': int(len(sgdf)), 'n_countries': int(sgdf['iso_code'].nunique()),
            'lambda': float(sem.betas[name_to_idx['lambda'], 0]), 'lambda_p_value': float(sem.z_stat[name_to_idx['lambda']][1])
        })
    sem_table = pd.DataFrame(sem_rows)
    ols_compare = pd.DataFrame(extract_ols_rows(ols_fit, std, 'M4 country-FE OLS, country-clustered SE', M4_predictors))
    ols_compare = ols_compare[['model','variable','coef_log_points_per_sd','cluster_se','p_value','stars','pct_change_per_sd','n_obs','n_countries']].rename(columns={'cluster_se':'std_error'})
    spatial_compare = pd.concat([ols_compare, sem_table[ols_compare.columns]], ignore_index=True, sort=False)
    spatial_compare.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_02_spatial_error_sensitivity_M4.csv', index=False)

    fig, ax = plt.subplots(figsize=(5.5, 4.8), constrained_layout=True)
    yy = sgdf['fe_residual'].values
    wy = wk.sparse @ yy
    ax.scatter(yy, wy, s=16, alpha=.55, color='#4c78a8', edgecolor='none')
    slope, intercept = np.polyfit(yy, wy, 1)
    xs = np.linspace(np.nanmin(yy), np.nanmax(yy), 100)
    ax.plot(xs, intercept + slope * xs, color='#d95f02', linewidth=1.3)
    ax.axhline(0, color='#777777', linewidth=.8)
    ax.axvline(0, color='#777777', linewidth=.8)
    ax.set_xlabel('M4 country-FE residual')
    ax.set_ylabel('Spatially lagged residual (KNN k=6)')
    ax.set_title(f'Moran scatterplot (I = {moran_k.I:.3f}, p = {moran_k.p_sim:.3f})')
    fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_02_M4_spatial_residual_moran_scatter.png', bbox_inches='tight')
    plt.show()

    spatial_status = {'status': 'completed', 'n_spatial_obs': int(len(sgdf)), 'moran_knn6_I': float(moran_k.I), 'moran_knn6_p': float(moran_k.p_sim), 'spatial_error_lambda': float(sem.betas[name_to_idx['lambda'], 0]), 'lambda_p': float(sem.z_stat[name_to_idx['lambda']][1])}
except Exception as e:
    spatial_status = {'status': 'failed', 'error': repr(e)}
    pd.DataFrame([spatial_status]).to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_02_spatial_error_status.csv', index=False)
spatial_status


In [ ]:

# 3. Fractional logit robustness for sufficiency_gap and sufficiency_deficit: global and regional M4.
frac_rows=[]
frac_summary=[]
for outcome in ['sufficiency_gap', 'sufficiency_deficit']:
    d = std.dropna(subset=[outcome] + ['z_' + v for v in M4_predictors] + ['iso_code']).copy()
    fit = fit_fractional_logit(d, outcome, M4_predictors)
    frac_rows.extend(extract_frac_rows(fit, d, 'M4_global_fractional_logit', outcome, M4_predictors, region='Global'))
    frac_summary.append({'outcome': outcome, 'scope': 'global', 'region': 'Global', 'n_obs': int(fit.nobs), 'n_countries': int(d['iso_code'].nunique()), 'aic': float(fit.aic), 'bic_llf': float(getattr(fit, 'bic_llf', np.nan)), 'status': 'ok'})
    for region, dreg in d.groupby('analytical_region'):
        if dreg['iso_code'].nunique() < 3 or len(dreg) < 30:
            frac_summary.append({'outcome': outcome, 'scope': 'region', 'region': region, 'n_obs': len(dreg), 'n_countries': dreg['iso_code'].nunique(), 'status': 'skipped'})
            continue
        rfit = fit_fractional_logit(dreg, outcome, M4_predictors)
        frac_rows.extend(extract_frac_rows(rfit, dreg, 'M4_region_fractional_logit', outcome, M4_predictors, region=region))
        frac_summary.append({'outcome': outcome, 'scope': 'region', 'region': region, 'n_obs': int(rfit.nobs), 'n_countries': int(dreg['iso_code'].nunique()), 'aic': float(rfit.aic), 'bic_llf': float(getattr(rfit, 'bic_llf', np.nan)), 'status': 'ok'})
frac = pd.DataFrame(frac_rows)
frac_summary = pd.DataFrame(frac_summary)
frac.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_03_fractional_logit_gap_deficit_M4_AME.csv', index=False)
frac_summary.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_03_fractional_logit_gap_deficit_M4_summary.csv', index=False)

plot = frac[(frac['variable'].isin(core_predictors)) & (frac['region'].eq('Global'))].copy()
plot['variable_label'] = plot['variable'].map(label_map)
fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
sns.pointplot(data=plot, x='AME_per_sd', y='variable_label', hue='outcome', dodge=.45, join=False, errorbar=None, ax=ax)
ax.axvline(0, color='#333333', lw=.8)
ax.set_xlabel('Average marginal effect per 1 SD increase')
ax.set_ylabel('')
ax.set_title('Global fractional logit robustness, M4')
fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_03_global_fractional_logit_core_AME.png', bbox_inches='tight')
plt.show()
frac[(frac['variable'].isin(['female_share','education'])) & (frac['p_value'] < 0.1)].sort_values(['outcome','region','variable'])


In [ ]:

# 4. Complete M4 interaction results for log sufficiency ratio.
interaction_pairs = [
    ('rural', 'SES_raw', 'Rurality x SES'),
    ('rural', 'education', 'Rurality x Education'),
    ('female_share', 'SES_raw', 'Female share x SES'),
    ('female_share', 'education', 'Female share x Education'),
    ('SES_raw', 'education', 'SES x Education'),
    ('age', 'female_share', 'Age x Female share'),
    ('SES_raw', 'Hhsize', 'SES x Household size'),
    ('female_share', 'rural', 'Female share x Rurality'),
]
interaction_rows=[]
slope_rows=[]
for x, z, label in interaction_pairs:
    predictors = list(dict.fromkeys(M4_predictors + [x, z]))
    d = std.dropna(subset=['log_sufficiency_ratio', 'iso_code'] + ['z_' + v for v in predictors]).copy()
    term = f'z_{x}:z_{z}'
    rhs = ' + '.join(['z_' + v for v in predictors] + [term, 'C(iso_code)'])
    fit = smf.ols('log_sufficiency_ratio ~ ' + rhs, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['iso_code']})
    ci = fit.conf_int()
    b=float(fit.params[term]); se=float(fit.bse[term]); p=float(fit.pvalues[term])
    lo, hi = ci.loc[term].tolist()
    interaction_rows.append({
        'outcome': 'log_sufficiency_ratio', 'model': 'M4_targeted_interaction_country_FE_clustered',
        'interaction': label, 'x': x, 'moderator': z,
        'coef_interaction_log_points': b, 'cluster_se': se, 'ci_low': float(lo), 'ci_high': float(hi),
        'p_value': p, 'stars_raw': star(p), 'n_obs': int(fit.nobs), 'n_countries': int(d['iso_code'].nunique()),
        'included_predictors': ', '.join(predictors)
    })
    cov = fit.cov_params()
    for focal, mod in [(x, z), (z, x)]:
        focal_term = 'z_' + focal
        int_term = term
        for level_name, mod_value in [('low_-1SD', -1.0), ('mean_0SD', 0.0), ('high_+1SD', 1.0)]:
            slope = float(fit.params[focal_term] + fit.params[int_term] * mod_value)
            var_slope = float(cov.loc[focal_term, focal_term] + (mod_value ** 2) * cov.loc[int_term, int_term] + 2 * mod_value * cov.loc[focal_term, int_term])
            se_slope = np.sqrt(max(var_slope, 0))
            z_stat = slope / se_slope if se_slope > 0 else np.nan
            p_slope = 2 * (1 - stats.norm.cdf(abs(z_stat))) if np.isfinite(z_stat) else np.nan
            slope_rows.append({
                'outcome': 'log_sufficiency_ratio', 'model': 'M4_targeted_interaction_country_FE_clustered',
                'interaction': label, 'focal_variable': focal, 'moderator_variable': mod,
                'moderator_level': level_name, 'moderator_z_value': mod_value,
                'slope_log_points_per_sd': slope, 'se': se_slope,
                'ci_low': slope - 1.96 * se_slope, 'ci_high': slope + 1.96 * se_slope,
                'p_value': p_slope, 'stars': star(p_slope), 'pct_change_per_sd': pct_change(slope)
            })
interaction = pd.DataFrame(interaction_rows)
rejected, qvals = fdrcorrection(interaction['p_value'].values, alpha=0.05)
interaction['q_value_fdr_across_8_tests'] = qvals
interaction['fdr_reject_0_05'] = rejected
interaction['stars_fdr'] = interaction['q_value_fdr_across_8_tests'].apply(star)
slopes = pd.DataFrame(slope_rows)
interaction.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_04_M4_interaction_terms_all_8.csv', index=False)
slopes.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_04_M4_interaction_simple_slopes_all_8.csv', index=False)

fig, ax = plt.subplots(figsize=(7.5, 4.8), constrained_layout=True)
plot = interaction.sort_values('coef_interaction_log_points')
y = np.arange(len(plot))
ax.errorbar(plot['coef_interaction_log_points'], y, xerr=[plot['coef_interaction_log_points'] - plot['ci_low'], plot['ci_high'] - plot['coef_interaction_log_points']], fmt='none', ecolor='0.35', capsize=2)
ax.scatter(plot['coef_interaction_log_points'], y, s=48, color='#7a5195')
ax.axvline(0, color='#333333', lw=.8)
ax.set_yticks(y)
ax.set_yticklabels(plot['interaction'])
ax.set_xlabel('Interaction coefficient, log points')
ax.set_title('Pre-specified M4 interactions; FDR across 8 tests')
for yy, (_, row) in enumerate(plot.iterrows()):
    txt = row['stars_raw'] if row['stars_raw'] else ''
    if txt:
        ax.text(row['ci_high'] + .01, yy, txt, va='center', fontsize=9)
fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_04_M4_interaction_terms_all_8.png', bbox_inches='tight')
plt.show()
interaction.sort_values('p_value')


## 5. Region-Specific Interaction Exploration

This section estimates the eight pre-specified M4 interaction terms separately within each analytical region. Models retain country fixed effects within region and cluster standard errors by country. Because this creates many tests and some regions have few countries, these estimates are exploratory. The table reports both FDR correction within each region's eight tests and FDR correction across all region-interaction tests.

In [ ]:

# 5. Exploratory region-specific M4 interaction models.
region_interaction_rows=[]
region_slope_rows=[]
region_interaction_status=[]
for region, dreg0 in std.groupby('analytical_region'):
    dreg0 = dreg0.copy()
    n_obs = len(dreg0)
    n_countries = dreg0['iso_code'].nunique()
    if n_countries < 5 or n_obs < 50:
        for x, z, label in interaction_pairs:
            region_interaction_status.append({
                'region': region, 'interaction': label, 'status': 'skipped_too_few_clusters_or_observations',
                'n_obs': n_obs, 'n_countries': n_countries
            })
        continue
    for x, z, label in interaction_pairs:
        predictors = list(dict.fromkeys(M4_predictors + [x, z]))
        needed = ['log_sufficiency_ratio', 'iso_code'] + ['z_' + v for v in predictors]
        dreg = dreg0.dropna(subset=needed).copy()
        term = f'z_{x}:z_{z}'
        rhs = ' + '.join(['z_' + v for v in predictors] + [term, 'C(iso_code)'])
        try:
            fit = smf.ols('log_sufficiency_ratio ~ ' + rhs, data=dreg).fit(cov_type='cluster', cov_kwds={'groups': dreg['iso_code']})
            ci = fit.conf_int()
            b=float(fit.params[term]); se=float(fit.bse[term]); p=float(fit.pvalues[term])
            lo, hi = ci.loc[term].tolist()
            region_interaction_rows.append({
                'outcome': 'log_sufficiency_ratio',
                'model': 'M4_region_targeted_interaction_country_FE_clustered',
                'region': region,
                'interaction': label,
                'x': x,
                'moderator': z,
                'coef_interaction_log_points': b,
                'cluster_se': se,
                'ci_low': float(lo),
                'ci_high': float(hi),
                'p_value': p,
                'stars_raw': star(p),
                'n_obs': int(fit.nobs),
                'n_countries': int(dreg['iso_code'].nunique()),
                'r_squared': float(fit.rsquared),
                'adj_r_squared': float(fit.rsquared_adj),
                'included_predictors': ', '.join(predictors)
            })
            region_interaction_status.append({
                'region': region, 'interaction': label, 'status': 'ok',
                'n_obs': int(fit.nobs), 'n_countries': int(dreg['iso_code'].nunique())
            })

            cov = fit.cov_params()
            for focal, mod in [(x, z), (z, x)]:
                focal_term = 'z_' + focal
                int_term = term
                for level_name, mod_value in [('low_-1SD', -1.0), ('mean_0SD', 0.0), ('high_+1SD', 1.0)]:
                    slope = float(fit.params[focal_term] + fit.params[int_term] * mod_value)
                    var_slope = float(cov.loc[focal_term, focal_term] + (mod_value ** 2) * cov.loc[int_term, int_term] + 2 * mod_value * cov.loc[focal_term, int_term])
                    se_slope = np.sqrt(max(var_slope, 0))
                    z_stat = slope / se_slope if se_slope > 0 else np.nan
                    p_slope = 2 * (1 - stats.norm.cdf(abs(z_stat))) if np.isfinite(z_stat) else np.nan
                    region_slope_rows.append({
                        'outcome': 'log_sufficiency_ratio',
                        'model': 'M4_region_targeted_interaction_country_FE_clustered',
                        'region': region,
                        'interaction': label,
                        'focal_variable': focal,
                        'moderator_variable': mod,
                        'moderator_level': level_name,
                        'moderator_z_value': mod_value,
                        'slope_log_points_per_sd': slope,
                        'se': se_slope,
                        'ci_low': slope - 1.96 * se_slope,
                        'ci_high': slope + 1.96 * se_slope,
                        'p_value': p_slope,
                        'stars': star(p_slope),
                        'pct_change_per_sd': pct_change(slope),
                        'n_obs': int(fit.nobs),
                        'n_countries': int(dreg['iso_code'].nunique())
                    })
        except Exception as e:
            region_interaction_status.append({
                'region': region, 'interaction': label, 'status': 'failed',
                'error': repr(e), 'n_obs': len(dreg), 'n_countries': dreg['iso_code'].nunique()
            })

region_interactions = pd.DataFrame(region_interaction_rows)
if not region_interactions.empty:
    # FDR within each region across the 8 pre-specified tests.
    for region, idx in region_interactions.groupby('region').groups.items():
        rejected, qvals = fdrcorrection(region_interactions.loc[idx, 'p_value'].values, alpha=0.05)
        region_interactions.loc[idx, 'q_value_fdr_within_region_8_tests'] = qvals
        region_interactions.loc[idx, 'fdr_within_region_reject_0_05'] = rejected
    # FDR across all region-interaction tests as a stricter family-level diagnostic.
    rejected_all, qvals_all = fdrcorrection(region_interactions['p_value'].values, alpha=0.05)
    region_interactions['q_value_fdr_all_region_interactions'] = qvals_all
    region_interactions['fdr_all_region_interactions_reject_0_05'] = rejected_all
    region_interactions['stars_fdr_within_region'] = region_interactions['q_value_fdr_within_region_8_tests'].apply(star)
    region_interactions['stars_fdr_all'] = region_interactions['q_value_fdr_all_region_interactions'].apply(star)

region_interaction_slopes = pd.DataFrame(region_slope_rows)
region_interaction_status = pd.DataFrame(region_interaction_status)
region_interactions.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_05_M4_region_interaction_terms_all_8.csv', index=False)
region_interaction_slopes.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_05_M4_region_interaction_simple_slopes_all_8.csv', index=False)
region_interaction_status.to_csv(SI_TABLE_DIR / 'Table_SI_RQ3_05_M4_region_interaction_status.csv', index=False)

if not region_interactions.empty:
    fig, ax = plt.subplots(figsize=(10.5, 5.8), constrained_layout=True)
    heat = region_interactions.pivot_table(index='interaction', columns='region', values='q_value_fdr_within_region_8_tests')
    sns.heatmap(heat, annot=True, fmt='.2f', cmap='viridis_r', vmin=0, vmax=0.25, cbar_kws={'label': 'Within-region FDR q-value'}, ax=ax)
    ax.set_title('Region-specific M4 interactions: FDR q-values within each region')
    ax.set_xlabel('')
    ax.set_ylabel('')
    fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_05_region_interaction_fdr_heatmap_M4.png', bbox_inches='tight')
    plt.show()

    sig_plot = region_interactions[region_interactions['p_value'] < 0.1].copy().sort_values(['p_value'])
    if not sig_plot.empty:
        sig_plot['label'] = sig_plot['region'] + ' | ' + sig_plot['interaction']
        fig, ax = plt.subplots(figsize=(8.5, max(3.8, 0.35 * len(sig_plot))), constrained_layout=True)
        y = np.arange(len(sig_plot))
        ax.errorbar(sig_plot['coef_interaction_log_points'], y,
                    xerr=[sig_plot['coef_interaction_log_points'] - sig_plot['ci_low'], sig_plot['ci_high'] - sig_plot['coef_interaction_log_points']],
                    fmt='none', ecolor='0.35', capsize=2)
        ax.scatter(sig_plot['coef_interaction_log_points'], y, s=45, color='#2f6f73')
        ax.axvline(0, color='#333333', lw=.8)
        ax.set_yticks(y)
        ax.set_yticklabels(sig_plot['label'])
        ax.set_xlabel('Interaction coefficient, log points')
        ax.set_title('Region-specific M4 interactions with raw p < 0.10')
        for yy, (_, row) in enumerate(sig_plot.iterrows()):
            tag = row['stars_raw'] if row['stars_raw'] else '†'
            ax.text(row['ci_high'] + .015, yy, tag, va='center', fontsize=9)
        fig.savefig(SI_FIG_DIR / 'Figure_SI_RQ3_05_region_interactions_raw_p_lt_0_10_M4.png', bbox_inches='tight')
        plt.show()

region_interactions.sort_values('p_value').head(30) if not region_interactions.empty else region_interaction_status


In [ ]:

manifest = {
    'notebook': str(PROJECT / 'RQ3_SI_clean.ipynb'),
    'data': str(DATA_PATH),
    'si_dir': str(SI_DIR),
    'notes': {
        'interaction_fdr_family': 'The eight pre-specified M4 interaction tests are corrected together using Benjamini-Hochberg FDR.',
        'spatial_sensitivity': 'KNN k=6 spatial error model is included because residual Moran I is significant and prior diagnostics favored spatial error over spatial lag.',
        'region_interactions': 'Region-specific M4 interactions are exploratory; country fixed effects are retained within each region and FDR is reported within-region and across all region-interaction tests.'
    },
    'sample_n': int(len(std)),
    'countries': int(std['iso_code'].nunique()),
    'spatial_status': spatial_status,
    'tables': sorted([str(p) for p in SI_TABLE_DIR.glob('Table_SI_RQ3_*.csv')]),
    'figures': sorted([str(p) for p in SI_FIG_DIR.glob('Figure_SI_RQ3_*.png')]),
}
(PROCESS_DIR / 'RQ3_SI_clean_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
